# Live trading Turtle Mean reversion 30 Min

Este cuaderno implementa la versión en vivo realizada en backtest en el cuaderno en Jupyter. Guarda el Journal de la jornada con el estado inicial, el monitoreo de MAE/R para cada posición y los cierres junto con el PnL. A diferencia del backtesting este cuaderno resume la ejecución a través de los módulos dirigidos por eventos:
* streaming.py : Conecta el Websocket de precios en vivo de IBKR para monitorear precios y data de usuario.
* event/event.py : Diccionario de eventos que incluye eventos de mercado (nuevos ticks) señales, calces , cierres y eventos de portafolio
* portfolio: implementa la lógica de portafolio , administración de posiciones , sizing , upsizing , downsizing, riesgo a nivel portafolio (si  multiactivos) estado de la cuenta (en local) , ordenes enviadas sin ejecución , niveles de calce etc. Es el módulo más complejo.
* strategy : Guarda la lógica de las estrategias y emite eventos señal al adminostrador de portafolio
* execution : Lanza ordenes a IBKR, Order Management System, reconcilia ordenes y posiciones , administra stops y condicionales. Administra la ejecución y mensajes de IBKR
* trading : Conecta todos los módulos y administra la cola de eventos.

Esta construcción originalmente se implementó para forex y acciones y se ha acondicionado particularmente para la prueba técnica propuesta por Sun Valley.  

# Imports y conexiones con IBKR


In [ ]:
# pip install ib_async pandas numpy
import json, os, queue, sys, threading, time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ib_async import IB, util

# ── Los módulos del motor. Toda la lógica vive aquí, no en el notebook ──────
from qsCrypto.ibkr import config
from qsCrypto.data.streaming import IBKRPriceHandler
from qsCrypto.strategy.strategy import MeanReversionFadeStrategy
from qsCrypto.portfolio.portfolio import IBKRFuturesPortfolio
from qsCrypto.execution.execution import create_execution_handler
from qsCrypto.trading.trading import trade

JOURNAL_DIR = ROOT / "journals"; JOURNAL_DIR.mkdir(exist_ok=True)
STATE_DIR = ROOT / "output"; STATE_DIR.mkdir(exist_ok=True)

# Para el cuaderno
def _in_jupyter():
    try:
        return get_ipython().__class__.__name__ == "ZMQInteractiveShell"
    except NameError:
        return False

if _in_jupyter():
    util.startLoop()          # sin esto connect() muere: el kernel ya tiene bucle

ib = IB()
# RequestTimeout viene a 0 y ib_async hace `if timeout:` — CERO SIGNIFICA SIN
# LÍMITE. Una petición sin respuesta colgaría el kernel para siempre y dejaría
# la posición sin vigilar.
ib.RequestTimeout = 20

IB_ERRORS = [] # Manejo de errores devueltos por la API
def _on_ib_error(reqId, code_, msg, contract):
    if code_ not in (2104, 2106, 2158, 2119):
        IB_ERRORS.append((datetime.now(timezone.utc).isoformat(), reqId, code_, msg))
        print(f"  [TWS {code_}] {msg}")

ib.connect(config.IB_HOST, config.IB_PORT, clientId=config.IB_CLIENT_ID,
           timeout=15, readonly=False)
ib.errorEvent += _on_ib_error

ACCOUNT = config.IB_ACCOUNT or ib.managedAccounts()[0]
ACCOUNTS = {a: ("master" if a.upper().startswith("DF") else "trading")
            for a in ib.managedAccounts()}
summary = {v.tag: v.value for v in ib.accountSummary(ACCOUNT)
           if v.tag in ("NetLiquidation", "AvailableFunds", "BuyingPower")}
EQUITY = Decimal(summary.get("NetLiquidation", "100000"))

print(f"  Conectado a {config.IB_HOST}:{config.IB_PORT} (clientId={config.IB_CLIENT_ID})")
print(f"  Cuenta operada : {ACCOUNT}")
print(f"  Cuentas        : {ACCOUNTS}")
print(f"  Equity         : {EQUITY:,.2f}")


  Conectado a 127.0.0.1:7496 (clientId=17)
  Cuenta operada : DU496961
  Cuentas        : {'DF496956': 'master', 'DU496957': 'trading', 'DU496958': 'trading', 'DU496959': 'trading', 'DU496960': 'trading', 'DU496961': 'trading'}
  Equity         : 351,598.67


Error 1300, reqId -1: Se ha restablecido el Socket Port y esta conexi\u00f3n dejar\u00e1 de funcionar. Vuelva a conectarse en el puerto nuevo.7496
Peer closed connection.


  [TWS 1300] Se ha restablecido el Socket Port y esta conexi\u00f3n dejar\u00e1 de funcionar. Vuelva a conectarse en el puerto nuevo.7496


## 1. Qué dijo el backtest, antes de arrancar nada

Esta implementación reproduce la estrategia de `MeanReversion_IBKR_CME_Bitcoin.ipynb`. Ese
notebook se dictamina a sí mismo **8 de 12 criterios** y concluye literalmente *"No desplegar
capital con esta evidencia"*. Los cuatro criterios que falla:

| Criterio | Exigido | Medido |
|---|---|---|
| Muestra | ≥ 2 años | **1,34 años** |
| Folds OOS | ≥ 8 | **6** (el t-test no tiene potencia) |
| Apuestas independientes | ≥ 8 | **n_eff = 1,00** |
| Deflated Sharpe | > 0,95 | **0,374** |

Sobre el DSR el backtest es explícito: el Sharpe seleccionado **no supera lo que cabría esperar
del mejor de esa rejilla sin ningún edge**. Con `n_eff = 1` un drawdown del 50% no es cola, es el
escenario central.

**Esto es un ejercicio técnico sobre cuenta paper.** El objetivo es que la estrategia corra por
los módulos dirigidos por eventos y que el MAE/MFE sea medible en vivo, no ganar dinero.

### El cierre temprano por MAE no puede dispararse

El backtest calibra **MAE\* = 1,55 N**, y opera un stop de **1 N**. Como `R = stop_n · N`, el
umbral de MAE cae **fuera** del stop: el stop del bróker llega siempre antes. El MAE se mide, se
registra y se muestra en el monitor, pero con `STOP_N = 1.0` **no cierra nada**. Subir `STOP_N`
haría que quepa, a cambio de abandonar el único punto de la curva de sensibilidad con Sharpe alto
(1,86 a 1N frente a 0,31 a 3N). El parámetro queda expuesto abajo.

In [2]:
# ─────────── PARÁMETROS, DECLARADOS ANTES DE ABRIR NADA ───────────
SYMBOLS       = ["MBT"]      # Micro Bitcoin CME, 0.1 BTC/contrato
EXCHANGE      = "CME"
CURRENCY      = "USD"
BAR_SIZE      = "30 mins"    # la granularidad del backtest
USE_RTH       = True
WHAT_TO_SHOW  = "TRADES"

# Señal (celda 4 y 24 del backtest)
ENTRY_BARS    = 20           # canal Donchian de entrada
ATR_PERIOD    = 20           # N = EMA-20 del True Range
SIGNAL_MODE   = "fade"       # se opera CONTRA la ruptura
REQUIRE_CONF  = True         # la barra t+1 debe cerrar dentro del canal
CONFIRM_MODE  = "inside"
ALLOW_SHORT   = True

# Riesgo y barreras
RISK_PER_UNIT = 0.01         # 1% del equity por unidad
STOP_N        = 1.0          # R = 1N. Ver §1: con esto el cierre por MAE no cabe.
ADD_N         = 0.5          # escalera de adds cada ½N
MAX_UNITS     = 4
MAX_HOLD_BARS = 10           # barrera vertical: 10 barras de 30 min = 5 h
TP_MODE       = "mid_channel"  # objetivo en el centro del canal, congelado en t
MAX_GROSS_LEV = 4.0
MAX_DIR_UNITS = 12
MAX_CORR_UNITS = 6
USE_DD_RULE   = True

WARMUP_BARS   = 120          # ≥ ATR_PERIOD + ENTRY_BARS, si no se opera a ciegas
HEARTBEAT     = 0.5          # cadencia del bucle de eventos
MAE_STAR_N    = 1.55         # medido en el backtest, solo para el informe

print(f"  Señal   : fade de ruptura {ENTRY_BARS} barras, confirmación={REQUIRE_CONF}")
print(f"  Riesgo  : stop {STOP_N:.0f}N = 1R | objetivo {TP_MODE} | vertical {MAX_HOLD_BARS} barras")
print(f"  Escalera: hasta {MAX_UNITS} unidades cada {ADD_N}N")
if MAE_STAR_N > STOP_N:
    print(f"\n  ⚠ MAE* = {MAE_STAR_N:.2f}N > stop {STOP_N:.2f}N ({MAE_STAR_N/STOP_N:.2f}R):")
    print("    el cierre temprano por MAE NO puede dispararse antes que el stop.")
    print("    Se mide y se registra, pero no cierra. Subir STOP_N lo haría caber.")


  Señal   : fade de ruptura 20 barras, confirmación=True
  Riesgo  : stop 1N = 1R | objetivo mid_channel | vertical 10 barras
  Escalera: hasta 4 unidades cada 0.5N

  ⚠ MAE* = 1.55N > stop 1.00N (1.55R):
    el cierre temprano por MAE NO puede dispararse antes que el stop.
    Se mide y se registra, pero no cierra. Subir STOP_N lo haría caber.


## 2. Cableado del motor

El flujo es el del README y no se altera aquí:

```
trading (cola)  →  data       emite BarEvent (barra cerrada) y TickEvent (bid/ask)
                →  strategy   consume BAR, emite SignalEvent
                →  portfolio  consume SIGNAL, dimensiona, emite OrderEvent
                →  execution  consume ORDER, manda bracket a IBKR, emite FillEvent
                →  portfolio  consume FILL, reancla al precio real, emite PortfolioEvent
                →  trading    enruta el PortfolioEvent al monitor
```

Dos hilos, como en `trading/trading.py`: uno para el bucle de eventos y otro para el streaming.
El notebook **no contiene lógica de estrategia**; solo construye los componentes y observa.

In [ ]:
# ── data ────────────────────────────────────────────────────────────────────
events = queue.Queue()

prices = IBKRPriceHandler(
    SYMBOLS, events, ib=ib, exchange=EXCHANGE, currency=CURRENCY,
    bar_size=BAR_SIZE, use_rth=USE_RTH, what_to_show=WHAT_TO_SHOW,
    warmup_bars=WARMUP_BARS, account=ACCOUNT,
)
prices.warmup()
SPECS = prices.specs
for s in SYMBOLS:
    h = prices.history[s]
    print(f"  {s}: {prices.specs[s]['localSymbol']}  mult={prices.specs[s]['multiplier']} "
          f"tick={prices.specs[s]['minTick']}  {len(h)} barras  "
          f"({h.index[0]:%m-%d %H:%M} → {h.index[-1]:%m-%d %H:%M})")

# ── strategy ────────────────────────────────────────────────────────────────
strategy = MeanReversionFadeStrategy(
    SYMBOLS, 
    events, 
    price_handler=prices, 
    entry_bars=ENTRY_BARS,
    atr_period=ATR_PERIOD, 
    signal_mode=SIGNAL_MODE,
    require_confirmation=REQUIRE_CONF, 
    confirm_mode=CONFIRM_MODE,
    allow_short=ALLOW_SHORT,
)

# ── execution ───────────────────────────────────────────────────────────────
execution = create_execution_handler(
    broker="ibkr",
    events_queue=events,
    ib=ib,
    contracts=prices.contracts, 
    specs=SPECS, account=ACCOUNT,
)

# ── portfolio ───────────────────────────────────────────────────────────────
portfolio = IBKRFuturesPortfolio(
    prices, 
    events, 
    home_currency=CURRENCY, 
    equity=EQUITY,
    risk_per_trade=Decimal(str(RISK_PER_UNIT)), 
    backtest=False, 
    specs=SPECS,
    stop_n=STOP_N, 
    add_n=ADD_N, 
    max_units=MAX_UNITS,
    max_hold_bars=MAX_HOLD_BARS, 
    tp_mode=TP_MODE,
    max_gross_lev=MAX_GROSS_LEV,
    max_dir_units=MAX_DIR_UNITS,
    max_corr_units=MAX_CORR_UNITS, 
    use_dd_rule=USE_DD_RULE,
    execution=execution, strategy=strategy,
)

n_now = strategy._n(strategy.history[SYMBOLS[0]])
qty_now = portfolio.calc_risk_position_size(SYMBOLS[0], n=n_now)
print(f"\n  N actual: {n_now:,.1f} puntos → 1R = {STOP_N*n_now:,.1f}")
print(f"  Unidad  : {qty_now} contratos → riesgo "
      f"${qty_now*STOP_N*n_now*SPECS[SYMBOLS[0]]['multiplier']:,.0f} "
      f"({RISK_PER_UNIT:.0%} del equity por construcción)")


marketDataType=3 (no es tiempo real): la MAE se mide sobre precios viejos y queda infravalorada.


  MBT: MBTU6  mult=0.1 tick=5.0  165 barras  (08-21 13:30 → 09-04 20:30)

  N actual: 424.1 puntos → 1R = 424.1
  Unidad  : 82 contratos → riesgo $3,478 (1% del equity por construcción)


## 3. Journal de la jornada

Una línea por decisión, append-only y con flush. Si el proceso muere, lo escrito hasta ese
instante sigue siendo la verdad reconstruible. El monitor recibe los `PortfolioEvent` que emite
el portfolio en cada cambio de estado, así que no tiene que hurgar en sus estructuras internas
mientras el hilo de precios las está mutando.

In [4]:
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
JOURNAL = JOURNAL_DIR / f"mr_intraday_{'_'.join(SYMBOLS)}_{SESSION_ID}.jsonl"
STATE = STATE_DIR / f"live_state_mr_{'_'.join(SYMBOLS)}.json"


class Monitor(object):
    """Recibe PortfolioEvent, escribe el journal y guarda la última foto."""

    def __init__(self, journal_path, state_path):
        self.journal_path = journal_path
        self.state_path = state_path
        self.last = []
        self.history = []
        self.lock = threading.Lock()

    def write(self, evento, **extra):
        row = {"ts": datetime.now(timezone.utc).isoformat(),
               "session": SESSION_ID, "evento": evento}
        row.update(extra)
        with self.lock:
            with open(self.journal_path, "a") as f:
                f.write(json.dumps(row, default=str) + "\n")
                f.flush()
        return row

    def record(self, event):
        """Lo llama trade() al recibir un PORTFOLIO."""
        with self.lock:
            self.last = event.snapshot
        self.history.append((datetime.now(timezone.utc), event.reason,
                             len(event.snapshot)))
        self.write("portfolio", reason=event.reason, balance=event.balance,
                   positions=event.snapshot)
        # Escritura atómica: un lector nunca ve un JSON a medias.
        tmp = self.state_path.with_suffix(".tmp")
        with open(tmp, "w") as f:
            json.dump({"ts": datetime.now(timezone.utc).isoformat(),
                       "session": SESSION_ID, "reason": event.reason,
                       "balance": event.balance, "positions": event.snapshot},
                      f, indent=2, default=str)
        os.replace(tmp, self.state_path)


monitor = Monitor(JOURNAL, STATE)
monitor.write("session_start", symbols=SYMBOLS, equity=float(EQUITY),
              account=ACCOUNT, stop_n=STOP_N, entry_bars=ENTRY_BARS,
              risk=RISK_PER_UNIT, mae_star_n=MAE_STAR_N,
              mae_fits_in_stop=bool(MAE_STAR_N <= STOP_N),
              specs={s: {k: v for k, v in SPECS[s].items() if k != "expiry"}
                     for s in SYMBOLS})
print(f"  Journal : {JOURNAL}")
print(f"  Estado  : {STATE}")


  Journal : /Users/diegoochoa/Projects/qsCrypto/journals/mr_intraday_MBT_20260907T222808Z.jsonl
  Estado  : /Users/diegoochoa/Projects/qsCrypto/output/live_state_mr_MBT.json


## 4. Arrancar la sesión

⚠️ **Esta celda abre posiciones reales en la cuenta paper.** `RUN_LIVE = False` deja todo montado
sin enviar nada, que es como conviene arrancar la primera vez.

In [5]:
RUN_LIVE = True       # ponlo a True para operar

if not RUN_LIVE:
    print("  RUN_LIVE=False — el motor no arranca. Nada se envía a IBKR.")
else:
    trade_thread = threading.Thread(
        target=trade,
        args=(events, strategy, portfolio, execution, HEARTBEAT, monitor),
        daemon=True, name="trade-loop",
    )
    price_thread = threading.Thread(
        target=prices.stream_to_queue, daemon=True, name="ibkr-stream",
    )
    trade_thread.start()
    price_thread.start()
    print("  Motor arrancado. Hilos:", [t.name for t in (trade_thread, price_thread)])
    print("  Ejecuta la celda del monitor cuantas veces quieras.")


  Motor arrancado. Hilos: ['trade-loop', 'ibkr-stream']
  Ejecuta la celda del monitor cuantas veces quieras.


## 5. Monitor de posiciones abiertas — MAE/MFE contra su stop

Una fila por posición viva. Las columnas que importan y por qué:

- **`mae_r` / `mfe_r`** — excursión adversa y favorable en unidades de R, donde `R = |entrada −
  stop inicial|`. Es la lectura que pide el enunciado y la única comparable entre instrumentos
  con volatilidades distintas.
- **`mae_budget`** — fracción del presupuesto de MAE consumida (`mae_r` sobre el umbral de
  cierre). Con `STOP_N = 1.0` el umbral está fuera del stop, así que esta columna nunca llega a
  1,00: el stop cierra antes. Es precisamente el hecho que hay que ver.
- **`dist_stop_r`** — cuánto queda hasta el stop, en R.
- **`pnl_r`** — resultado no realizado en R.

Debajo, los trades ya cerrados con su excursión final, cosechada por el portfolio antes de
destruir la posición.

In [6]:
def show_monitor():
    with monitor.lock:
        rows = list(monitor.last) if monitor.last else portfolio.snapshot()
    if not rows:
        print("  Sin posiciones abiertas.")
    else:
        df = pd.DataFrame(rows)
        df["mae_budget"] = df["mae_r"] / (MAE_STAR_N / STOP_N)
        cols = ["pair", "side", "qty", "units", "entry", "price", "stop", "n",
                "r_unit", "pnl_r", "mae_r", "mae_price", "mae_budget",
                "mfe_r", "dist_stop_r", "bars", "pnl_base"]
        display(df[[c for c in cols if c in df.columns]].round(5))
        for _, r in df.iterrows():
            print(f"  {r['pair']} {r['side']} x{int(r['qty'])} | "
                  f"P&L {r['pnl_r']:+.2f}R | MAE {r['mae_r']:.2f}R "
                  f"({r['mae_budget']:.0%} del umbral) | MFE {r['mfe_r']:.2f}R | "
                  f"stop a {r['dist_stop_r']:.2f}R")

    closed = portfolio.closed_excursions
    print(f"\n  Trades cerrados: {len(closed)}")
    if closed:
        cdf = pd.DataFrame(closed)
        cols = [c for c in ["currency_pair", "position_type", "units",
                            "entry_anchor", "exit_price", "pnl", "mae_r",
                            "mfe_r", "mae_price", "mfe_price"] if c in cdf.columns]
        display(cdf[cols].round(5))
        if "mae_r" in cdf and cdf["mae_r"].notna().any():
            win = cdf["pnl"] > 0
            print(f"  MAE media ganadores: {cdf.loc[win,'mae_r'].mean():.2f}R | "
                  f"perdedores: {cdf.loc[~win,'mae_r'].mean():.2f}R")

    print(f"\n  Rechazos estrategia: {strategy.rejected}")
    print(f"  Rechazos portfolio : {portfolio.rejected}")
    if IB_ERRORS:
        print(f"  Errores TWS        : {IB_ERRORS[-3:]}")


show_monitor()


  Sin posiciones abiertas.

  Trades cerrados: 0

  Rechazos estrategia: {'sin_confirmar': 0, 'plana': 0, 'sin_n': 0, 'posicion_abierta': 0}
  Rechazos portfolio : {'qty_cero': 0, 'apalancamiento': 0, 'topes': 0}
  Errores TWS        : [('2026-09-07T22:28:36.598371+00:00', 314, 10197, 'No hay datos de mercado durante sesiones competidoras  en directo')]


## 6. Cierre de la jornada

Cancela lo vivo, aplana y cierra la conexión. `flatten()` cancela **antes** de aplanar: al revés,
el stop adjunto seguiría vivo tras el cierre y quedaría como una orden huérfana capaz de abrir
posición contraria.

In [ ]:
def kill_all(reason="fin_de_jornada"):
    for s in SYMBOLS:
        try:
            flat = execution.flatten(s, reason=reason)
            monitor.write("kill", symbol=s, flat=flat, reason=reason)
            print(f"  {s}: aplanado {flat:+d}")
        except Exception as exc:
            print(f"  {s}: fallo al aplanar — {exc}")


# kill_all()          # descomenta para aplanar todo ahora mismo

# monitor.write("session_end", closed=len(portfolio.closed_excursions),
#               balance=float(portfolio.balance))
# prices.stop(); ib.disconnect(); print("  Sesión cerrada.")


Exception in thread trade-loop:
Traceback (most recent call last):
  File "/Users/diegoochoa/.pyenv/versions/3.11.9/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/Users/diegoochoa/.pyenv/versions/3.11.9/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/diegoochoa/Projects/qsCrypto/qsCrypto/trading/trading.py", line 39, in trade
    logger.info("Received new tick event: %s", event)
    ^^^^^^
NameError: name 'logger' is not defined
Exception in thread ibkr-stream:
Traceback (most recent call last):
  File "/Users/diegoochoa/Projects/qsCrypto/.venv/lib/python3.11/site-packages/ib_async/util.py", line 360, in run
    result = loop.run_until_complete(task)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/diegoochoa/Projects/qsCrypto/.venv/lib/python3.11/site-packages/nest_asyncio.py", line 98, in run_until_complete
    return f.result()
           ^^^^^^^^^^
  File "/Users/diegoochoa/.p